# NFL Survivor Pick'em - FPI Data Analysis

This notebook demonstrates how to use the FPI Loader Utility to analyze Football Power Index data for NFL survivor pick'em strategy.


In [ ]:
# Import the FPI loader utility functions
from utils.fpi_loader import load_fpi_data, get_team_fpi, get_fpi_rankings, find_latest_fpi_file
from utils.matchup_analyzer import (
    analyze_weekly_matchups, 
    get_weekly_picks, 
    get_top_picks_by_week,
    save_matchup_analysis
)


In [ ]:
# 2. Load all FPI data
fpi_data = load_fpi_data()
if fpi_data is None:
    print("Failed to load FPI data!")
else:
    print(f"Loaded FPI data for {len(fpi_data)} teams")
    print()


In [ ]:
# 3. Show top 10 teams by FPI
print("Top 10 Teams by FPI:")
print("-" * 30)
rankings = get_fpi_rankings()
if rankings is not None:
    for _, row in rankings.head(10).iterrows():
        print(f"{row['Rank']:2d}. {row['Team']:<25} {row['FPI']:5.1f}")
print()


In [ ]:
# 5. Look up specific teams
teams_to_check = ["Philadelphia Eagles", "Kansas City Chiefs", "New Orleans Saints"]
print("Specific Team FPI Ratings:")
print("-" * 30)
for team in teams_to_check:
    fpi = get_team_fpi(team)
    if fpi is not None:
        print(f"{team:<25} {fpi:5.1f}")
print()


In [ ]:
# Import matchup analyzer functions
from utils.matchup_analyzer import (
    analyze_weekly_matchups, 
    get_weekly_picks, 
    get_top_picks_by_week,
    save_matchup_analysis
)


In [ ]:
# Import matchup analyzer
from utils.matchup_analyzer import analyze_weekly_matchups, save_matchup_analysis

# Create/update the FPI matchup analysis file
analysis = analyze_weekly_matchups()
if analysis is not None:
    success = save_matchup_analysis(analysis, "data/fpi_matchup_analysis.csv")
    if success:
        print("FPI matchup analysis file created/updated successfully!")
    else:
        print("Failed to save analysis file")
else:
    print("Failed to analyze matchups")

# show the dataframe that was created
analysis

In [ ]:
# Import team weekly analyzer
from utils.team_weekly_analyzer import create_team_weekly_fpi_table, save_team_weekly_fpi_table

# Create/update the team-weekly FPI table (excluding already picked teams)
team_weekly_table = create_team_weekly_fpi_table(picks_file="data/picks_tracking.csv")
if team_weekly_table is not None:
    success = save_team_weekly_fpi_table(team_weekly_table, "data/team_weekly_fpi.csv")
    if success:
        print("Team-weekly FPI table created/updated successfully!")
    else:
        print("Failed to save team-weekly table")
else:
    print("Failed to create team-weekly table")


In [ ]:
# from analysis, select all records for week 1
analysis[analysis['Week'] == 'Week 1']
# from analysis, select all records for week 2


In [ ]:
# Import team weekly analyzer
from utils.team_weekly_analyzer import create_team_weekly_fpi_table, save_team_weekly_fpi_table

# Create/update the team-weekly FPI table (excluding already picked teams)
team_weekly_table = create_team_weekly_fpi_table(picks_file="data/picks_tracking.csv")
if team_weekly_table is not None:
    success = save_team_weekly_fpi_table(team_weekly_table, "data/team_weekly_fpi.csv")
    if success:
        print("Team-weekly FPI table created/updated successfully!")
    else:
        print("Failed to save team-weekly table")
else:
    print("Failed to create team-weekly table")

team_weekly_table

In [ ]:
# Create filtered team_weekly_table excluding already picked teams and weeks
import pandas as pd

# Read the picks tracking file
picks_df = pd.read_csv("data/picks_tracking.csv")

# Get the teams that have already been picked (non-empty Entry_Pick values)
picked_teams = picks_df[picks_df['Entry_Pick'].notna() & (picks_df['Entry_Pick'] != '')]['Entry_Pick'].tolist()
print(f"Already picked teams: {picked_teams}")

# Get the weeks that have already been picked (corresponding to picked teams)
picked_weeks = picks_df[picks_df['Entry_Pick'].notna() & (picks_df['Entry_Pick'] != '')]['Week'].tolist()
picked_week_columns = [f"Week {week}" for week in picked_weeks]
print(f"Already picked weeks: {picked_week_columns}")

# Create a copy of the original team_weekly_table
filtered_team_weekly_table = team_weekly_table.copy()

# Remove already picked teams (rows)
filtered_team_weekly_table = filtered_team_weekly_table[~filtered_team_weekly_table.index.isin(picked_teams)]
print(f"\nRemoved {len(picked_teams)} teams. Remaining teams: {len(filtered_team_weekly_table)}")

# Remove already picked weeks (columns)
columns_to_remove = [col for col in picked_week_columns if col in filtered_team_weekly_table.columns]
filtered_team_weekly_table = filtered_team_weekly_table.drop(columns=columns_to_remove)
print(f"Removed {len(columns_to_remove)} week columns. Remaining weeks: {len(filtered_team_weekly_table.columns)}")

print(f"\nFiltered table shape: {filtered_team_weekly_table.shape}")
filtered_team_weekly_table.head()


In [ ]:
# Maximin Optimization for Survivor Pick Selection
# Goal: Maximize the minimum weekly FPI value across all weeks
import numpy as np
from itertools import combinations
import pandas as pd

def maximin_survivor_picks(fpi_table):
    """
    Find the optimal survivor picks using maximin optimization.
    
    Args:
        fpi_table: DataFrame with teams as rows and weeks as columns
        
    Returns:
        dict: {week: (team, fpi_value), ...}, min_value
    """
    
    # Convert to numpy for easier manipulation, handling NaN values
    teams = fpi_table.index.tolist()
    weeks = fpi_table.columns.tolist()
    
    print(f"Optimizing picks for {len(teams)} teams across {len(weeks)} weeks...")
    print("Ignoring NaN values (bye weeks)...")
    
    best_min_value = float('-inf')
    best_solution = None
    
    # We need to try different combinations since we have more weeks (16) than remaining teams (30)
    # But we can only use each team once, so we need to select 16 teams from 30
    
    from itertools import permutations
    
    # Since we have 30 teams and 16 weeks, we need to select 16 teams and assign them to weeks
    # This is computationally intensive, so let's use a greedy approach with some optimization
    
    return greedy_maximin_picks(fpi_table)

def greedy_maximin_picks(fpi_table):
    """
    Greedy approach to maximin optimization with backtracking for better solutions.
    """
    teams = fpi_table.index.tolist()
    weeks = fpi_table.columns.tolist()
    
    # Start with greedy assignment
    used_teams = set()
    picks = {}
    weekly_values = []
    
    # Sort weeks by how constrained they are (fewer good options)
    week_constraints = {}
    for week in weeks:
        valid_values = fpi_table[week].dropna()
        week_constraints[week] = len(valid_values)
    
    # Process most constrained weeks first
    sorted_weeks = sorted(weeks, key=lambda w: week_constraints[w])
    
    print("Processing weeks in order of constraint (most constrained first):")
    for week in sorted_weeks:
        print(f"  {week}: {week_constraints[week]} available teams")
    
    for week in sorted_weeks:
        # Get available teams for this week (not used and not NaN)
        available_data = fpi_table[week].dropna()
        available_teams = [team for team in available_data.index if team not in used_teams]
        
        if not available_teams:
            print(f"ERROR: No available teams for {week}!")
            return None, float('-inf')
        
        # Among available teams, pick the one that maximizes the current minimum
        best_team = None
        best_value = float('-inf')
        
        for team in available_teams:
            fpi_value = fpi_table.loc[team, week]
            
            # Calculate what the minimum would be if we pick this team
            temp_values = weekly_values + [fpi_value]
            temp_min = min(temp_values)
            
            if temp_min > best_value:
                best_value = temp_min
                best_team = team
        
        # Make the pick
        picks[week] = (best_team, fpi_table.loc[best_team, week])
        used_teams.add(best_team)
        weekly_values.append(fpi_table.loc[best_team, week])
        
        print(f"{week}: {best_team} (FPI: {fpi_table.loc[best_team, week]:.1f})")
    
    final_min = min(weekly_values)
    print(f"\nFinal minimum weekly FPI: {final_min:.1f}")
    print(f"Weekly FPI values: {[round(v, 1) for v in weekly_values]}")
    
    return picks, final_min

# Run the optimization
print("="*60)
print("SURVIVOR PICK OPTIMIZATION - MAXIMIN APPROACH")
print("="*60)

optimal_picks, min_fpi = maximin_survivor_picks(filtered_team_weekly_table)

if optimal_picks:
    print(f"\nOPTIMAL PICKS SUMMARY:")
    print("-" * 40)
    for week in filtered_team_weekly_table.columns:
        team, fpi = optimal_picks[week]
        print(f"{week:8}: {team:25} (FPI: {fpi:5.1f})")
    
    print(f"\nMinimum weekly FPI achieved: {min_fpi:.1f}")
else:
    print("Optimization failed!")


In [ ]:
# Create summary table and save results
if optimal_picks:
    # Create a summary DataFrame
    summary_data = []
    for week in filtered_team_weekly_table.columns:
        team, fpi = optimal_picks[week]
        summary_data.append({
            'Week': week,
            'Team': team,
            'FPI': fpi
        })
    
    optimal_picks_df = pd.DataFrame(summary_data)
    
    # Save to CSV
    optimal_picks_df.to_csv("data/optimal_survivor_picks.csv", index=False)
    print("Optimal picks saved to: data/optimal_survivor_picks.csv")
    
    # Display the summary table
    print(f"\nOPTIMAL SURVIVOR PICKS TABLE:")
    print("="*50)
    display(optimal_picks_df)
    
    # Show some statistics
    fpi_values = [fpi for _, fpi in optimal_picks.values()]
    print(f"\nSTATISTICS:")
    print(f"Minimum FPI: {min(fpi_values):.1f}")
    print(f"Maximum FPI: {max(fpi_values):.1f}")
    print(f"Average FPI: {np.mean(fpi_values):.1f}")
    print(f"Standard Deviation: {np.std(fpi_values):.1f}")
    
    # Show how this compares to a naive approach (just picking best available each week)
    print(f"\nCOMPARISON TO NAIVE APPROACH:")
    print("-" * 30)
    
    # Naive approach: pick best available team each week
    naive_picks = {}
    naive_used = set()
    naive_values = []
    
    for week in filtered_team_weekly_table.columns:
        available_data = filtered_team_weekly_table[week].dropna()
        available_teams = [team for team in available_data.index if team not in naive_used]
        
        if available_teams:
            # Pick team with highest FPI this week
            best_naive_team = available_data[available_teams].idxmax()
            best_naive_fpi = available_data[best_naive_team]
            
            naive_picks[week] = (best_naive_team, best_naive_fpi)
            naive_used.add(best_naive_team)
            naive_values.append(best_naive_fpi)
    
    print(f"Naive min FPI: {min(naive_values):.1f}")
    print(f"Optimal min FPI: {min(fpi_values):.1f}")
    print(f"Improvement: +{min(fpi_values) - min(naive_values):.1f}")
    
else:
    print("No optimal solution found!")


In [ ]:
# Test the modified team_weekly_analyzer with picks exclusion
from utils.team_weekly_analyzer import create_team_weekly_fpi_table, save_team_weekly_fpi_table

# Create team-weekly FPI table excluding already picked teams
print("Creating team-weekly FPI table with picked teams excluded...")
print("="*60)

team_weekly_table_filtered = create_team_weekly_fpi_table(picks_file="data/picks_tracking.csv")

if team_weekly_table_filtered is not None:
    print(f"\nFiltered table created successfully!")
    print(f"Shape: {team_weekly_table_filtered.shape}")
    print(f"Teams: {len(team_weekly_table_filtered.index)}")
    print(f"Weeks: {len(team_weekly_table_filtered.columns)}")
    
    # Show first few teams
    print(f"\nFirst 5 teams:")
    print(team_weekly_table_filtered.head().index.tolist())
    
    # Save the filtered table
    success = save_team_weekly_fpi_table(team_weekly_table_filtered, "data/team_weekly_fpi_filtered.csv")
    if success:
        print(f"\nFiltered table saved to: data/team_weekly_fpi_filtered.csv")
    else:
        print(f"Failed to save filtered table")
        
    # Show the filtered table
    team_weekly_table_filtered.head()
else:
    print("Failed to create filtered team-weekly table")


In [ ]:
# Restart kernel and reload modules to pick up changes
import importlib
import sys

# Force reload of the team_weekly_analyzer module
if 'utils.team_weekly_analyzer' in sys.modules:
    importlib.reload(sys.modules['utils.team_weekly_analyzer'])

# Import the updated functions
from utils.team_weekly_analyzer import create_team_weekly_fpi_table, save_team_weekly_fpi_table

print("Modules reloaded successfully!")


In [ ]:
# Test the updated function signature
print("Testing updated create_team_weekly_fpi_table function...")
print("="*60)

# Test with picks file parameter
team_weekly_table_filtered = create_team_weekly_fpi_table(picks_file="data/picks_tracking.csv")

if team_weekly_table_filtered is not None:
    print(f"\nFiltered table created successfully!")
    print(f"Shape: {team_weekly_table_filtered.shape}")
    print(f"Teams: {len(team_weekly_table_filtered.index)}")
    print(f"Weeks: {len(team_weekly_table_filtered.columns)}")
    
    # Show first few teams
    print(f"\nFirst 5 teams:")
    print(team_weekly_table_filtered.head().index.tolist())
    
    # Save the filtered table
    success = save_team_weekly_fpi_table(team_weekly_table_filtered, "data/team_weekly_fpi_filtered.csv")
    if success:
        print(f"\nFiltered table saved to: data/team_weekly_fpi_filtered.csv")
    else:
        print(f"Failed to save filtered table")
        
    # Show the filtered table
    team_weekly_table_filtered.head()
else:
    print("Failed to create filtered team-weekly table")


In [11]:
# ============================================================================
# REMAINING TEAMS ANALYSIS - Which entries have the best teams left?
# ============================================================================

import pandas as pd
import numpy as np
from utils.fpi_loader import load_fpi_data, get_team_fpi

print("="*80)
print("SURVIVOR POOL - REMAINING TEAMS ANALYSIS")
print("="*80)

# 1. Load FPI data ONCE
print("\n1. Loading FPI data...")
fpi_data = load_fpi_data()
if fpi_data is None:
    print("ERROR: Failed to load FPI data!")
else:
    print(f"   ✓ Loaded FPI data for {len(fpi_data)} teams")
    
# Create a dictionary for fast FPI lookups (team_name -> FPI value)
fpi_dict = dict(zip(fpi_data['Team'], fpi_data['FPI']))
print(f"   ✓ Created FPI lookup dictionary with {len(fpi_dict)} teams")

# 2. Load abbreviation mapping
print("\n2. Loading team abbreviation mapping...")
abbrev_df = pd.read_csv("data/abbreviation_mapping.csv")
abbrev_to_name = dict(zip(abbrev_df['team_abbreviation'], abbrev_df['team_name']))
print(f"   ✓ Loaded {len(abbrev_to_name)} team mappings")

# 3. Load all picks
print("\n3. Loading all picks data...")
picks_df = pd.read_csv("data/all_picks.csv", encoding='latin-1')
print(f"   ✓ Loaded picks for {len(picks_df)} entries")

# 4. Get all 32 NFL teams
all_nfl_teams = set(abbrev_df['team_name'].tolist())
print(f"\n4. NFL teams: {len(all_nfl_teams)} total teams")

# 4b. Calculate weeks remaining
TOTAL_WEEKS = 18
CURRENT_WEEK = 5  # We've completed Week 5
WEEKS_REMAINING = TOTAL_WEEKS - CURRENT_WEEK
print(f"\n4b. Weeks remaining: {WEEKS_REMAINING} (Week {CURRENT_WEEK+1} through Week {TOTAL_WEEKS})")
print(f"   → Will analyze TOP {WEEKS_REMAINING} remaining teams for each entry")

# 5. Process each entry
print("\n5. Processing each entry's picks...")
print("-" * 80)

week_columns = [col for col in picks_df.columns if col.startswith('Week ')]
results = []

for idx, row in picks_df.iterrows():
    entry_name = row['Name']
    
    # Get all picks for this entry (ignore empty and 'X')
    picked_abbrevs = []
    for week_col in week_columns:
        pick = row[week_col]
        if pd.notna(pick) and pick != '' and pick != 'X':
            picked_abbrevs.append(pick.lower())
    
    # Convert abbreviations to full team names
    picked_teams = []
    for abbrev in picked_abbrevs:
        if abbrev in abbrev_to_name:
            picked_teams.append(abbrev_to_name[abbrev])
        else:
            print(f"   WARNING: Unknown abbreviation '{abbrev}' for {entry_name}")
    
    # Calculate remaining teams
    remaining_teams = all_nfl_teams - set(picked_teams)
    
    # Get FPI values for remaining teams (use dictionary lookup - much faster!)
    remaining_fpis = []
    for team in remaining_teams:
        fpi = fpi_dict.get(team)  # Direct dictionary lookup instead of calling get_team_fpi()
        if fpi is not None:
            remaining_fpis.append(fpi)
    
    # Sort FPIs descending and take only top N teams (where N = weeks remaining)
    remaining_fpis_sorted = sorted(remaining_fpis, reverse=True)
    top_n_fpis = remaining_fpis_sorted[:WEEKS_REMAINING]  # Only use top N teams
    
    # Calculate metrics on TOP N teams only
    if top_n_fpis:
        avg_fpi = np.mean(top_n_fpis)
        max_fpi = np.max(top_n_fpis)
        min_fpi = np.min(top_n_fpis)
        median_fpi = np.median(top_n_fpis)
        std_fpi = np.std(top_n_fpis)
    else:
        avg_fpi = max_fpi = min_fpi = median_fpi = std_fpi = 0
    
    results.append({
        'Entry_Name': entry_name,
        'Teams_Picked': len(picked_teams),
        'Teams_Remaining': len(remaining_teams),
        'Avg_FPI_Remaining': avg_fpi,
        'Max_FPI_Remaining': max_fpi,
        'Min_FPI_Remaining': min_fpi,
        'Median_FPI_Remaining': median_fpi,
        'Std_FPI_Remaining': std_fpi,
        'Picked_Teams': ', '.join(sorted(picked_teams))
    })

# Create results DataFrame
results_df = pd.DataFrame(results)

# Sort by average FPI (descending - best first)
results_df = results_df.sort_values('Avg_FPI_Remaining', ascending=False)
results_df['Rank'] = range(1, len(results_df) + 1)

# Calculate percentile (1% = best, 100% = worst)
results_df['Percentile'] = (results_df['Rank'] / len(results_df) * 100).round(1)

# Reorder columns
results_df = results_df[['Rank', 'Percentile', 'Entry_Name', 'Teams_Picked', 'Teams_Remaining', 
                         'Avg_FPI_Remaining', 'Max_FPI_Remaining', 'Min_FPI_Remaining',
                         'Median_FPI_Remaining', 'Std_FPI_Remaining', 'Picked_Teams']]

print(f"\n   ✓ Processed {len(results_df)} entries")

# 6. Display summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Weeks remaining: {WEEKS_REMAINING}")
print(f"Average teams picked per entry: {results_df['Teams_Picked'].mean():.1f}")
print(f"Average teams remaining per entry: {results_df['Teams_Remaining'].mean():.1f}")
print(f"\n→ Analyzing TOP {WEEKS_REMAINING} teams for each entry:")
print(f"Average FPI of top {WEEKS_REMAINING} teams: {results_df['Avg_FPI_Remaining'].mean():.2f}")
print(f"Best average FPI (top {WEEKS_REMAINING}): {results_df['Avg_FPI_Remaining'].max():.2f}")
print(f"Worst average FPI (top {WEEKS_REMAINING}): {results_df['Avg_FPI_Remaining'].min():.2f}")

# 7. Display top 20 entries with best remaining teams
print("\n" + "="*80)
print(f"TOP 20 ENTRIES - BEST REMAINING TEAM POOLS (Top {WEEKS_REMAINING} teams by Avg FPI)")
print("="*80)
print(f"{'Rank':<6} {'Pct':<6} {'Entry Name':<35} {'Picked':<8} {'Remain':<8} {'Avg FPI':<10} {'Max FPI':<10} {'Min FPI':<10}")
print("-" * 80)

for _, row in results_df.head(20).iterrows():
    print(f"{row['Rank']:<6} {row['Percentile']:<6}% {row['Entry_Name']:<35} {row['Teams_Picked']:<8} "
          f"{row['Teams_Remaining']:<8} {row['Avg_FPI_Remaining']:<10.2f} "
          f"{row['Max_FPI_Remaining']:<10.2f} {row['Min_FPI_Remaining']:<10.2f}")

# 8. Display bottom 20 entries
print("\n" + "="*80)
print(f"BOTTOM 20 ENTRIES - WORST REMAINING TEAM POOLS (Top {WEEKS_REMAINING} teams by Avg FPI)")
print("="*80)
print(f"{'Rank':<6} {'Pct':<6} {'Entry Name':<35} {'Picked':<8} {'Remain':<8} {'Avg FPI':<10} {'Max FPI':<10} {'Min FPI':<10}")
print("-" * 80)

for _, row in results_df.tail(20).iterrows():
    print(f"{row['Rank']:<6} {row['Percentile']:<6}% {row['Entry_Name']:<35} {row['Teams_Picked']:<8} "
          f"{row['Teams_Remaining']:<8} {row['Avg_FPI_Remaining']:<10.2f} "
          f"{row['Max_FPI_Remaining']:<10.2f} {row['Min_FPI_Remaining']:<10.2f}")

# 9. Save to CSV
output_file = "data/entry_remaining_teams_analysis.csv"
results_df.to_csv(output_file, index=False)
print(f"\n✓ Full results saved to: {output_file}")

# Display the full dataframe
print("\n" + "="*80)
print("FULL RESULTS DATAFRAME")
print("="*80)
results_df


SURVIVOR POOL - REMAINING TEAMS ANALYSIS

1. Loading FPI data...
Found latest FPI file: week5_fpi.csv (Week 5)
Loaded FPI data: 32 teams
   ✓ Loaded FPI data for 32 teams
   ✓ Created FPI lookup dictionary with 32 teams

2. Loading team abbreviation mapping...
   ✓ Loaded 32 team mappings

3. Loading all picks data...
   ✓ Loaded picks for 2184 entries

4. NFL teams: 32 total teams

4b. Weeks remaining: 13 (Week 6 through Week 18)
   → Will analyze TOP 13 remaining teams for each entry

5. Processing each entry's picks...
--------------------------------------------------------------------------------

   ✓ Processed 2184 entries

SUMMARY STATISTICS
Weeks remaining: 13
Average teams picked per entry: 5.0
Average teams remaining per entry: 27.0

→ Analyzing TOP 13 teams for each entry:
Average FPI of top 13 teams: 2.59
Best average FPI (top 13): 3.39
Worst average FPI (top 13): 1.80

TOP 20 ENTRIES - BEST REMAINING TEAM POOLS (Top 13 teams by Avg FPI)
Rank   Pct    Entry Name           

,Rank,Percentile,Entry_Name,Teams_Picked,Teams_Remaining,Avg_FPI_Remaining,Max_FPI_Remaining,Min_FPI_Remaining,Median_FPI_Remaining,Std_FPI_Remaining,Picked_Teams
294,1,0.0,Phillip Carroll #2,5,27,3.392308,6.1,1.4,3.1,1.430149,"Arizona Cardinals, Jacksonville Jaguars, Miami..."
1740,2,0.1,Fred Shearer #3,5,27,3.392308,6.1,1.4,3.1,1.430149,"Arizona Cardinals, Cincinnati Bengals, Minneso..."
1754,3,0.1,Bailey Siewert #1,5,27,3.392308,6.1,1.4,3.1,1.430149,"Arizona Cardinals, Dallas Cowboys, Minnesota V..."
399,4,0.2,Heidi Dauner #2,5,27,3.392308,6.1,1.4,3.1,1.430149,"Arizona Cardinals, Dallas Cowboys, Minnesota V..."
999,5,0.2,Chenny Koulavong #3,5,27,3.353846,6.1,1.4,3.1,1.475761,"Arizona Cardinals, Carolina Panthers, Cincinna..."
...,...,...,...,...,...,...,...,...,...,...,...
297,2180,99.8,Andrew Chapirson #1,5,27,1.876923,4.5,-0.2,1.6,1.433207,"Buffalo Bills, Denver Broncos, Detroit Lions, ..."
2014,2181,99.9,Caralyn Visvader #2,5,27,1.823077,4.5,-0.2,1.6,1.355986,"Buffalo Bills, Denver Broncos, Detroit Lions, ..."
376,2182,99.9,Melissa Curley #1,5,27,1.823077,4.5,-0.2,1.6,1.355986,"Buffalo Bills, Denver Broncos, Detroit Lions, ..."
1758,2183,100.0,Jeffrey Silvers #1,5,27,1.800000,4.5,-0.2,1.6,1.336471,"Buffalo Bills, Detroit Lions, Kansas City Chie..."


In [ ]:
# ============================================================================
# DETAILED ENTRY EXPLORER - See remaining teams for specific entries
# ============================================================================

# ============================================================================
# DETAILED ENTRY EXPLORER - See remaining teams for specific entries
# ============================================================================

import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors

def show_entry_details(entry_name_search, save_pdf=True):
    """
    Show detailed breakdown of remaining teams for a specific entry.
    
    Args:
        entry_name_search: Full or partial entry name to search for
        save_pdf: Whether to save a PDF report (default: True)
    """
    # Find matching entries
    matching = results_df[results_df['Entry_Name'].str.contains(entry_name_search, case=False, na=False)]
    
    if len(matching) == 0:
        print(f"No entries found matching '{entry_name_search}'")
        return
    
    if len(matching) > 1:
        print(f"Found {len(matching)} entries matching '{entry_name_search}':")
        for _, row in matching.iterrows():
            print(f"  - {row['Entry_Name']} (Rank #{row['Rank']})")
        print("\nPlease be more specific or view first match below:")
        print("-" * 80)
    
    # Show details for first match
    entry = matching.iloc[0]
    
    print("\n" + "="*80)
    print(f"ENTRY DETAILS: {entry['Entry_Name']}")
    print("="*80)
    print(f"Overall Rank: #{entry['Rank']} out of {len(results_df)} entries")
    print(f"Percentile: {entry['Percentile']}% (1% = best, 100% = worst)")
    print(f"After Week {CURRENT_WEEK}")
    print(f"\nTop {WEEKS_REMAINING} Remaining Teams - FPI Statistics:")
    print(f"  (Metrics based on best {WEEKS_REMAINING} teams only)")
    print(f"  Average FPI:    {entry['Avg_FPI_Remaining']:6.2f}")
    print(f"  Median FPI:     {entry['Median_FPI_Remaining']:6.2f}")
    print(f"  Max FPI:        {entry['Max_FPI_Remaining']:6.2f}")
    print(f"  Min FPI:        {entry['Min_FPI_Remaining']:6.2f}")
    print(f"  Std Deviation:  {entry['Std_FPI_Remaining']:6.2f}")
    
    print(f"\n{'-'*80}")
    print("TEAMS ALREADY PICKED:")
    print(f"{'-'*80}")
    picked_teams = entry['Picked_Teams'].split(', ')
    for i, team in enumerate(picked_teams, 1):
        fpi = fpi_dict.get(team, 0)  # Use dictionary lookup
        print(f"  {i}. {team:<30} (FPI: {fpi:6.2f})")
    
    # Calculate remaining teams
    entry_picked_set = set(picked_teams)
    remaining = all_nfl_teams - entry_picked_set
    
    # Get FPI for ALL teams (remaining + picked) and sort
    all_teams_with_fpi = []
    for team in all_nfl_teams:
        fpi = fpi_dict.get(team)  # Use dictionary lookup
        if fpi is not None:
            all_teams_with_fpi.append((team, fpi))
    
    all_teams_with_fpi.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{'-'*80}")
    print(f"ALL TEAMS - Sorted by FPI (Picked teams shown crossed out):")
    print(f"{'-'*80}")
    print(f"{'Rank':<6} {'Team':<30} {'FPI':<10} {'Status':<15}")
    print(f"{'-'*80}")
    
    remaining_count = 0
    for i, (team, fpi) in enumerate(all_teams_with_fpi, 1):
        if team in entry_picked_set:
            # Team was already picked - show crossed out
            team_display = f"~~{team}~~"
            marker = "PICKED"
        else:
            # Team is remaining
            team_display = team
            remaining_count += 1
            marker = "top remaining" if remaining_count <= WEEKS_REMAINING else ""
        
        print(f"{i:<6} {team_display:<30} {fpi:<10.2f} {marker:<15}")
    
    # Save PDF if requested
    if save_pdf:
        try:
            # Create PDF directory if it doesn't exist
            pdf_dir = "data/pdf"
            os.makedirs(pdf_dir, exist_ok=True)
            
            # Create PDF filename (sanitize entry name)
            safe_name = "".join(c for c in entry['Entry_Name'] if c.isalnum() or c in (' ', '-', '_')).rstrip()
            safe_name = safe_name.replace(' ', '_')
            pdf_filename = f"{pdf_dir}/{safe_name}_Week{CURRENT_WEEK}_Report.pdf"
            
            # Create PDF document
            doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
            styles = getSampleStyleSheet()
            story = []
            
            # Title
            title_style = ParagraphStyle(
                'CustomTitle',
                parent=styles['Heading1'],
                fontSize=16,
                spaceAfter=30,
                alignment=1  # Center
            )
            story.append(Paragraph(f"Entry Analysis: {entry['Entry_Name']}", title_style))
            story.append(Spacer(1, 12))
            
            # Summary info
            summary_data = [
                ['Overall Rank:', f"#{entry['Rank']} out of {len(results_df)} entries"],
                ['Percentile:', f"{entry['Percentile']}% (1% = best, 100% = worst)"],
                ['After Week:', str(CURRENT_WEEK)],
                ['Teams Picked:', str(entry['Teams_Picked'])],
                ['Teams Remaining:', str(entry['Teams_Remaining'])]
            ]
            
            summary_table = Table(summary_data, colWidths=[2*inch, 3*inch])
            summary_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (0, -1), colors.lightgrey),
                ('TEXTCOLOR', (0, 0), (-1, -1), colors.black),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 0), (-1, -1), 10),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 12),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            story.append(summary_table)
            story.append(Spacer(1, 20))
            
            # FPI Statistics
            story.append(Paragraph(f"Top {WEEKS_REMAINING} Remaining Teams - FPI Statistics", styles['Heading2']))
            story.append(Paragraph(f"(Metrics based on best {WEEKS_REMAINING} teams only)", styles['Normal']))
            story.append(Spacer(1, 12))
            
            stats_data = [
                ['Average FPI:', f"{entry['Avg_FPI_Remaining']:.2f}"],
                ['Median FPI:', f"{entry['Median_FPI_Remaining']:.2f}"],
                ['Max FPI:', f"{entry['Max_FPI_Remaining']:.2f}"],
                ['Min FPI:', f"{entry['Min_FPI_Remaining']:.2f}"],
                ['Std Deviation:', f"{entry['Std_FPI_Remaining']:.2f}"]
            ]
            
            stats_table = Table(stats_data, colWidths=[2*inch, 1*inch])
            stats_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (0, -1), colors.lightblue),
                ('TEXTCOLOR', (0, 0), (-1, -1), colors.black),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 0), (-1, -1), 10),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 12),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            story.append(stats_table)
            story.append(Spacer(1, 20))
            
            # Teams table
            story.append(Paragraph("All Teams - Sorted by FPI", styles['Heading2']))
            story.append(Paragraph("(Picked teams shown with strikethrough)", styles['Normal']))
            story.append(Spacer(1, 12))
            
            # Prepare teams data for PDF
            teams_data = [['Rank', 'Team', 'FPI', 'Status']]
            remaining_count = 0
            
            for i, (team, fpi) in enumerate(all_teams_with_fpi, 1):
                if team in entry_picked_set:
                    team_display = f"{team} (PICKED)"
                    marker = "PICKED"
                else:
                    team_display = team
                    remaining_count += 1
                    marker = "top remaining" if remaining_count <= WEEKS_REMAINING else ""
                
                teams_data.append([str(i), team_display, f"{fpi:.2f}", marker])
            
            teams_table = Table(teams_data, colWidths=[0.5*inch, 2.5*inch, 0.8*inch, 1.2*inch])
            teams_table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
                ('FONTSIZE', (0, 0), (-1, -1), 8),
                ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
                ('GRID', (0, 0), (-1, -1), 1, colors.black),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.lightgrey])
            ]))
            story.append(teams_table)
            
            # Build PDF
            doc.build(story)
            print(f"\n✓ PDF report saved to: {pdf_filename}")
            
        except Exception as e:
            print(f"\n⚠️  Could not save PDF: {e}")
            print("   (PDF generation requires 'reportlab' package: pip install reportlab)")
    
    return matching

# Example usage - check top ranked entry
print("Example: Viewing details for the #1 ranked entry...")
print("="*80)
top_entry = results_df.iloc[0]['Entry_Name']
show_entry_details('fortier')

print("\n\n" + "="*80)
print("To explore other entries, call:")
print("  show_entry_details('entry_name_or_partial_match')")
print("="*80)


Example: Viewing details for the #1 ranked entry...

ENTRY DETAILS: Erik Fortier #2
Overall Rank: #120 out of 2184 entries
Percentile: 5.5% (1% = best, 100% = worst)
After Week 5

Top 13 Remaining Teams - FPI Statistics:
  (Metrics based on best 13 teams only)
  Average FPI:      3.04
  Median FPI:       3.10
  Max FPI:          6.10
  Min FPI:         -0.10
  Std Deviation:    1.86

--------------------------------------------------------------------------------
TEAMS ALREADY PICKED:
--------------------------------------------------------------------------------
  1. Arizona Cardinals              (FPI:  -1.70)
  2. Dallas Cowboys                 (FPI:   1.30)
  3. Houston Texans                 (FPI:   1.90)
  4. Indianapolis Colts             (FPI:   2.80)
  5. Seattle Seahawks               (FPI:   1.40)

--------------------------------------------------------------------------------
ALL TEAMS - Sorted by FPI (Picked teams shown crossed out):
-------------------------------------

In [ ]:
# ============================================================================
# VISUALIZATIONS - Compare entries by remaining pool quality
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

# Create a 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Survivor Pool - Top {WEEKS_REMAINING} Remaining Teams Analysis', fontsize=16, fontweight='bold')

# 1. Top 20 Entries - Average FPI Remaining
ax1 = axes[0, 0]
top_20 = results_df.head(20).sort_values('Avg_FPI_Remaining', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_20)))
ax1.barh(range(len(top_20)), top_20['Avg_FPI_Remaining'], color=colors)
ax1.set_yticks(range(len(top_20)))
ax1.set_yticklabels([name[:25] for name in top_20['Entry_Name']], fontsize=8)
ax1.set_xlabel(f'Average FPI of Top {WEEKS_REMAINING} Remaining Teams', fontweight='bold')
ax1.set_title(f'Top 20 Entries - Best Remaining Pools (Top {WEEKS_REMAINING} Teams)', fontweight='bold')
ax1.axvline(results_df['Avg_FPI_Remaining'].mean(), color='red', linestyle='--', 
            label=f'Overall Avg: {results_df["Avg_FPI_Remaining"].mean():.2f}')
ax1.legend()

# 2. Distribution of Average FPI Remaining
ax2 = axes[0, 1]
ax2.hist(results_df['Avg_FPI_Remaining'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(results_df['Avg_FPI_Remaining'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {results_df["Avg_FPI_Remaining"].mean():.2f}')
ax2.axvline(results_df['Avg_FPI_Remaining'].median(), color='green', linestyle='--', 
            linewidth=2, label=f'Median: {results_df["Avg_FPI_Remaining"].median():.2f}')
ax2.set_xlabel(f'Average FPI of Top {WEEKS_REMAINING} Remaining Teams', fontweight='bold')
ax2.set_ylabel('Number of Entries', fontweight='bold')
ax2.set_title(f'Distribution of Top {WEEKS_REMAINING} Team Pool Quality', fontweight='bold')
ax2.legend()

# 3. Teams Picked vs Avg FPI Remaining (scatter plot)
ax3 = axes[1, 0]
scatter = ax3.scatter(results_df['Teams_Picked'], results_df['Avg_FPI_Remaining'], 
                     c=results_df['Avg_FPI_Remaining'], cmap='RdYlGn', 
                     s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
ax3.set_xlabel('Number of Teams Already Picked', fontweight='bold')
ax3.set_ylabel(f'Average FPI of Top {WEEKS_REMAINING} Remaining Teams', fontweight='bold')
ax3.set_title(f'Teams Picked vs Top {WEEKS_REMAINING} Pool Quality', fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='Avg FPI')

# Add trend line
z = np.polyfit(results_df['Teams_Picked'], results_df['Avg_FPI_Remaining'], 1)
p = np.poly1d(z)
ax3.plot(results_df['Teams_Picked'], p(results_df['Teams_Picked']), 
         "r--", alpha=0.8, linewidth=2, label='Trend')
ax3.legend()

# 4. Top 15 - Compare Avg, Max, Min FPI
ax4 = axes[1, 1]
top_15 = results_df.head(15)
x = np.arange(len(top_15))
width = 0.25

bars1 = ax4.bar(x - width, top_15['Min_FPI_Remaining'], width, label='Min FPI', color='coral', alpha=0.8)
bars2 = ax4.bar(x, top_15['Avg_FPI_Remaining'], width, label='Avg FPI', color='steelblue', alpha=0.8)
bars3 = ax4.bar(x + width, top_15['Max_FPI_Remaining'], width, label='Max FPI', color='lightgreen', alpha=0.8)

ax4.set_xlabel('Entry (by Rank)', fontweight='bold')
ax4.set_ylabel('FPI Value', fontweight='bold')
ax4.set_title(f'Top 15 Entries - FPI Range (Top {WEEKS_REMAINING} Teams)', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([f"#{i+1}" for i in range(len(top_15))], fontsize=9)
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/remaining_teams_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved to: data/remaining_teams_analysis.png")
plt.show()

# Additional: Box plot showing spread of remaining FPIs for top 10 entries
fig, ax = plt.subplots(figsize=(14, 8))

top_10 = results_df.head(10)
box_data = []
labels = []

for _, entry in top_10.iterrows():
    picked_teams = entry['Picked_Teams'].split(', ')
    entry_picked_set = set(picked_teams)
    remaining = all_nfl_teams - entry_picked_set
    
    # Use dictionary lookup and take only top N teams
    remaining_fpis = [fpi_dict[team] for team in remaining if team in fpi_dict]
    remaining_fpis_sorted = sorted(remaining_fpis, reverse=True)
    top_n_fpis = remaining_fpis_sorted[:WEEKS_REMAINING]  # Only top N teams
    box_data.append(top_n_fpis)
    labels.append(f"#{entry['Rank']}\n{entry['Entry_Name'][:20]}")

bp = ax.boxplot(box_data, labels=labels, patch_artist=True, showmeans=True)

# Color the boxes
colors = plt.cm.RdYlGn(np.linspace(0.5, 0.9, len(box_data)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Entry', fontweight='bold', fontsize=12)
ax.set_ylabel(f'FPI Values of Top {WEEKS_REMAINING} Remaining Teams', fontweight='bold', fontsize=12)
ax.set_title(f'Top 10 Entries - Distribution of Top {WEEKS_REMAINING} Team FPIs', fontweight='bold', fontsize=14)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('data/top10_fpi_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Distribution plot saved to: data/top10_fpi_distribution.png")
plt.show()


In [ ]:
# Alternative approach - if reload doesn't work, try this:
try:
    team_weekly_table = create_team_weekly_fpi_table(data_folder="data", picks_file="data/picks_tracking.csv")
    print("Function call with explicit parameters successful!")
except TypeError as e:
    print(f"Error: {e}")
    print("Please restart the kernel or run the reload cell first.")
